# MCP Threat Hunting and Runtime Anomaly Detection

Run a complete offline hunt lifecycle: authorize evidence access, prove data readiness, execute deterministic and anomaly methods, preserve finding provenance, record analyst disposition, and peer-review a detection candidate. No SIEM, alert, incident, or containment action is created.

## Outcomes

You will inspect hunt grants, falsifiable hypotheses, data-quality gates, invariant drift, robust MAD scoring, low-and-slow behavior, sequence novelty, NetworkX authority graphs, analyst disposition, detection promotion, and labelled operational metrics.

## Authority boundary

A model or detector proposes a hypothesis, query, score, or finding. Trusted application code authorizes evidence access, validates provenance and quality, bounds analysis, preserves evidence, and governs promotion. An anomaly never authorizes containment.

In [ ]:
import runpy
from dataclasses import replace
from datetime import timedelta
module = runpy.run_path('lab.py')
NOW = module['NOW']
sorted(name for name in ('HuntAuthority', 'EvidencePreparer', 'HuntEngine', 'DetectionRegistry') if name in module)

## Scenario

The Acme support platform has reviewed one server artifact, capability contract, destination, operation sequence, and authority graph. The hunt asks whether runtime evidence contains an unreviewed artifact, capability, destination, delegation, or graph edge that current alert logic missed.

In [ ]:
authority, preparer, grant, hypothesis, engine = module['demo_components']()
(hypothesis.hypothesis_id, hypothesis.version, hypothesis.methods, grant.purpose)

## 1. Inspect the falsifiable hypothesis

A hunt is reproducible only when its statement, scope, sources, fields, methods, exclusions, validation, and promotion criteria are versioned. ATT&CK/ATLAS references organize coverage; they are not executable evidence.

In [ ]:
module['validate_hypothesis'](hypothesis)
{
    'statement': hypothesis.statement,
    'tenants': sorted(hypothesis.tenant_ids),
    'sources': sorted(hypothesis.required_sources),
    'fields': sorted(hypothesis.required_fields),
    'validation': hypothesis.validation_steps,
}

## 2. Authorize read scope first

The Ed25519-signed hunt grant binds hunter, role, purpose, tenant set, event kinds, time window, policy version, issuance, and expiry. It grants read scope only—never detection activation or response authority.

In [ ]:
authority.verify(grant, NOW)
{
    'hunter': grant.hunter_id, 'role': grant.hunter_role,
    'purpose': grant.purpose, 'tenants': sorted(grant.tenant_ids),
    'window': (grant.window_start, grant.window_end),
}

## 3. Prepare governed runtime evidence

One gateway event represents an approved call. One network event records a successful export to an unapproved destination. Both carry exact source, tenant, artifact, capability, trace, operation, event/observed/ingest time, and evidence digest.

In [ ]:
event_fixture = module['event_fixture']
EventKind = module['EventKind']
events = [
    event_fixture(event_id='EV-TOOL', source_id='source:gateway'),
    event_fixture(
        event_id='EV-EGRESS', source_id='source:network', kind=EventKind.NETWORK,
        action='ticket.export', destination='exfil.example', bytes_out=50_000,
        occurred_at=NOW + timedelta(seconds=2), observed_at=NOW + timedelta(seconds=3),
        ingested_at=NOW + timedelta(seconds=4), operation_id='OP-EXFIL',
    ),
]
[(e.event_id, e.source_id, e.kind, e.destination) for e in events]

In [ ]:
prepared = preparer.prepare(events, grant, hypothesis, NOW)
prepared.report

### Readiness before interpretation

The report names source coverage, field completeness, duplicates, invalid evidence, late events, and clock errors. An empty result is not clean when required sources or fields are absent.

In [ ]:
gap = preparer.prepare(events[:1], grant, hypothesis, NOW)
{'ready': gap.report.ready, 'coverage': gap.report.source_coverage, 'reasons': gap.report.reason_codes}

### Reflection

If a network sensor is absent, what can a zero-result egress query establish? Which operational signal distinguishes true zero activity from ingestion, parser, retention, authorization, or query failure?

## 4. Prefer deterministic invariants

When trusted inventory already defines the expected artifact, capability, destination, tenant, or delegation depth, compare directly. Machine learning adds no authority and can reduce explainability.

In [ ]:
invariant_findings = engine.invariant_hunt(prepared, hypothesis, NOW + timedelta(seconds=5))
[(f.finding_id, f.method, f.reason_codes, f.evidence_event_ids, f.confidence) for f in invariant_findings]

## 5. Use a versioned robust baseline

The transparent MAD method requires a preceding, contamination-reviewed population, a minimum sample count, finite non-negative values, non-zero dispersion, and a reviewed threshold. Its score is not a probability of attack.

In [ ]:
Baseline = module['BaselineContract']
baseline = Baseline(
    'BASE-BYTES-1', '1.0.0', 'tenant:acme', 'server:support', 'ticket.export',
    'bytes_out', NOW - timedelta(days=8), NOW - timedelta(days=1),
    (450, 480, 500, 510, 520, 540, 560), 5, 3.5, True,
)
mad_findings = engine.robust_mad_hunt(prepared, hypothesis, baseline, NOW + timedelta(seconds=5))
[(f.score, f.threshold, f.baseline_id, f.baseline_version) for f in mad_findings]

### Interpretation

The 50,000-byte export is far from reviewed history. Validate deployment changes, source accuracy, operation purpose, downstream effect, and baseline contamination. If MAD is zero, the lab returns indeterminate instead of adding an epsilon and fabricating an extreme score.

## 6. Hunt low-and-slow denials

A burst alert can miss probing distributed below its five-minute threshold. The hunt requires enough total denials over a minimum span while keeping every five-minute bucket below the burst limit.

In [ ]:
Outcome = module['Outcome']
slow_events = [events[1]]
for index in range(6):
    at = NOW - timedelta(minutes=30) + timedelta(minutes=6 * index)
    slow_events.append(event_fixture(
        event_id=f'EV-DENY-{index}', source_id='source:gateway', kind=EventKind.POLICY,
        policy_decision='deny', outcome=Outcome.DENIED, operation_id=f'OP-D{index}',
        occurred_at=at, observed_at=at + timedelta(seconds=1), ingested_at=at + timedelta(seconds=2),
    ))
slow_prepared = preparer.prepare(slow_events, grant, hypothesis, NOW)
[(f.entity_id, len(f.evidence_event_ids), f.reason_codes) for f in engine.low_and_slow_hunt(slow_prepared, hypothesis, NOW)]

## 7. Hunt novel operation sequences

A sequence groups same-tenant evidence by stable operation ID and orders it by event time. Novelty is low-confidence because valid releases, late events, and missing sources can produce a new order.

In [ ]:
sequence_events = [
    event_fixture(event_id='EV-S1', source_id='source:gateway', operation_id='OP-SEQ', action='ticket.read'),
    event_fixture(event_id='EV-S2', source_id='source:network', kind=EventKind.NETWORK, operation_id='OP-SEQ', action='ticket.export', occurred_at=NOW + timedelta(seconds=2), observed_at=NOW + timedelta(seconds=3), ingested_at=NOW + timedelta(seconds=4)),
]
sequence_prepared = preparer.prepare(sequence_events, grant, hypothesis, NOW)
[(f.entity_id, f.reason_codes, f.confidence) for f in engine.sequence_hunt(sequence_prepared, hypothesis, NOW + timedelta(seconds=5))]

## 8. Hunt authority and egress graph edges

NetworkX constructs a directed graph inside each authorized tenant. The lab compares principal-to-server and server-to-destination edges with reviewed architecture and cites the event that created every unexpected edge.

In [ ]:
graph_findings = engine.graph_hunt(prepared, hypothesis, NOW + timedelta(seconds=5))
[(f.entity_id, f.evidence_event_ids, f.reason_codes) for f in graph_findings]

## 9. Preserve candidate provenance

A candidate stores hypothesis and method, tenant/entity, exact evidence IDs/digest, reason codes, score/threshold, baseline identity/version, time, confidence, and application-owned disposition. Confidence is not probability of maliciousness.

In [ ]:
egress = next(f for f in invariant_findings if 'UNAPPROVED_DESTINATION' in f.reason_codes)
{
    'finding': egress.finding_id, 'hypothesis': (egress.hypothesis_id, egress.hypothesis_version),
    'evidence': egress.evidence_event_ids, 'digest': egress.evidence_digest,
    'confidence': egress.confidence, 'disposition': egress.disposition,
}

## 10. Require accountable analyst disposition

Supported means evidence supports the scoped behavior, not attacker intent. Benign and inconclusive remain evidence-preserving outcomes. A model cannot disposition a finding.

In [ ]:
supported, disposition = module['disposition_finding'](
    egress, 'analyst:oncall', 'threat_hunter', module['Disposition'].SUPPORTED,
    'trusted inventory and network evidence confirm unapproved egress',
    NOW + timedelta(seconds=6),
)
(supported.disposition, disposition.finding_digest == module['digest'](egress), disposition.evidence_event_ids)

## 11. Promote only to a test detection

The candidate binds supported findings, query and field contracts, window/grouping/threshold, expected false positives, labelled dataset metrics, and owner. A separate peer reviewer approves the exact digest.

In [ ]:
Candidate = module['DetectionCandidate']
candidate = Candidate(
    'DC-NB', 'MCP-HUNT-001', '1.0.0', 'Unapproved MCP egress',
    hypothesis.hypothesis_id, (supported.finding_id,),
    'kind=network AND destination NOT IN trusted_inventory.allowed_destinations',
    ('tenant_id', 'server_id', 'destination', 'event_id'), timedelta(minutes=5),
    ('tenant_id', 'server_id', 'destination'), None, hypothesis.exclusions,
    'dataset:mcp-egress-v1', 1.0, 1.0, 'team:detection-engineering',
)
candidate.state

In [ ]:
registry = module['DetectionRegistry']()
receipt = registry.issue_promotion(candidate, 'reviewer:peer', 'detection_reviewer', NOW + timedelta(seconds=7))
promoted = registry.register_test(candidate, receipt, NOW + timedelta(seconds=8), [supported])
(promoted.state, list(registry.rules), 'no containment API exists in this registry')

### Reflection

What additional target-SIEM replay, performance, cost, shadow deployment, runbook, monitoring, rollback, and release evidence is needed before `test` becomes `active`? Why is response authorization still a different workflow?

## 12. Evaluate security value and analyst cost

Use labelled episodes because one behavior can create many events/findings. Report precision, recall, event-time to finding, findings per scanned population, and analyst minutes per supported episode.

In [ ]:
Episode = module['LabelledEpisode']
episodes = [
    Episode('M1', True, ('F1',), NOW, NOW + timedelta(seconds=30), 10, 100),
    Episode('M2', True, (), NOW, None, 5, 100),
    Episode('V1', False, ('F2',), NOW, NOW + timedelta(seconds=5), 8, 400),
    Episode('V2', False, (), NOW, None, 2, 400),
]
module['hunt_metrics'](episodes)

## Production tool choices

Keep deterministic allowlist and policy hunts in explicit queries/rules. Use Sigma for portable source content, then test compiled queries in each backend. For statistical methods, compare transparent baselines with scikit-learn, PyOD, streaming libraries, or SIEM-native detectors such as OpenSearch RCF. Pin versions, feature contracts, training populations, seeds where supported, thresholds, drift behavior, and rollback.

## Exercises and final Reflection

1. Remove a required source and prove the result is `not ready`, not clean.
2. Compare a fixed policy threshold, MAD, and Isolation Forest on a labelled seasonal bytes-out dataset.
3. Add a denial -> credential change -> success temporal hunt and compile an equivalent Sigma rule.
4. Extend the graph with credentials, audiences, scopes, and resource tenants; prove tenant isolation.
5. Threat-model an LLM query assistant for invented fields, scope widening, injection, evidence suppression, and direct response attempts.

Reflection: for each output in this notebook, identify whether it is trusted evidence, a deterministic mismatch, a statistical signal, an analyst judgment, a peer-reviewed test artifact, or an authority-bearing decision.